In [ ]:
import __init__

#### MP-20 XRD ablations
- Dataset: Source [Materials Project](https://next-gen.materialsproject.org/)
- Ablation study on XRD conditioning and perplexity-based ranking for a model trained from scratch on MP-20

### Preprocess

In [ ]:
import pandas as pd
df = pd.read_parquet('/home/cyprien/Data_Gen/mp-20-data/all_data.parquet')
df

In [ ]:
!python _utils/_preprocessing/_calculate_theor_XRD.py \
    --input_parquet /home/cyprien/Data_Gen/mp-20-data/all_data.parquet \
    --output_parquet HF-databases/mp_20_pxrd/mp_20_pxrd.parquet \
    --num_workers 16

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/mp_20_pxrd/mp_20_pxrd.parquet \
    --output_parquet HF-databases/mp_20_pxrd/mp_20_pxrd_clean.parquet \
    --property_columns "['Condition Vector']" \
    --num_workers 16 \
    --filter_to 1024

In [ ]:
!python _utils/_preprocessing/_save_dataset_to_HF.py \
    --input_parquet HF-databases/mp_20_pxrd/mp_20_pxrd_clean.parquet \
    --output_parquet HF-databases/mp_20_pxrd/mp_20_pxrd.parquet \
    --save_hub

In [ ]:
!python _utils/_generating/make_prompts.py \
    --HF_dataset 'c-bone/mp_20_pxrd' \
    --split 'test' \
    --automatic \
    --output_parquet '_artifacts/mp-20-pxrd/mp-test_prompts.parquet' \
    --level 'level_3' \
    --condition_columns 'Condition Vector'
    # --remove_ref_columns

### Training

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/mp-20-pxrd-slider-scratch-opt.jsonc'

In [ ]:
import pandas as pd
import __init__
df = pd.read_parquet('_artifacts/mp-20-pxrd/mp-test_prompts.parquet')

# make a random subset of 1000
df = df.sample(1000, random_state=1).reset_index(drop=True)
df.to_parquet('_artifacts/mp-20-pxrd/mp-test_prompts_1k.parquet', index=False)
# make a no condition version
df['condition_vector'] = df['condition_vector'].apply(lambda x: ','.join(['-100'] * 40))
df.to_parquet('_artifacts/mp-20-pxrd/mp-test_prompts_1k_nocond.parquet', index=False)

### Ablations

#### 1. XRD conditions + perplexity scoring
- Sweep temperatures: top-p = 0.95, top-k = 10, T ∈ [0.5, 0.75, 1.0, 1.25, 1.5, 1.75]

In [ ]:
import __init__

In [ ]:
T = ['050', '075', '100', '125', '150', '175']

In [ ]:
# The reference table is published beside the dataset. Download it instead of rebuilding it.
import shutil
from huggingface_hub import hf_hub_download

shutil.copy(hf_hub_download("c-bone/mp_20_pxrd", "reference/mp-test_ref.parquet", repo_type="dataset"),
            "_artifacts/mp-20-pxrd/mp-test_ref.parquet")


In [ ]:
for temp in T:
    gen_config = f"_config_files/generation/conditional/xrd_studies/temp/mp-scratch-20perp-{temp}T_eval.jsonc"
    gen_parquet = f"_artifacts/mp-20-pxrd/temp/mp-20-scratch-20perp-{temp}T_gen.parquet"
    post_parquet = f"_artifacts/mp-20-pxrd/temp/mp-20-scratch-20perp-{temp}_post.parquet"
    metric_20perp_parquet = f"_artifacts/mp-20-pxrd/temp/mp-20-scratch-20perp-{temp}_metrics.parquet"
    metrics_1perp_parquet = f"_artifacts/mp-20-pxrd/temp/mp-20-scratch-1perp-{temp}_metrics.parquet"

    !python _utils/_generating/generate_CIFs.py --config {gen_config}

    !python _utils/_generating/postprocess.py \
    --input_parquet {gen_parquet} \
    --output_parquet {post_parquet} \
    --num_workers 32 

    !python _utils/_metrics/XRD_metrics.py \
    --input_parquet {post_parquet} \
    --num_gens 20 \
    --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
    --output_parquet {metric_20perp_parquet} \
    --num_workers 24 \
    --validity_check 'diffcsp'

    !python _utils/_metrics/XRD_metrics.py \
    --input_parquet {post_parquet} \
    --num_gens 1 \
    --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
    --output_parquet {metric_1perp_parquet} \
    --num_workers 24 \
    --validity_check 'diffcsp'

In [ ]:
import __init__
from _utils._notebook_utils.y_xrd_mp_20_ablations_utils import get_metrics_xrd
import pandas as pd

results = {}
for temp in T:
    # if temp == '050':
    #     continue
    metric_20perp_parquet = f"_artifacts/mp-20-pxrd/temp/mp-20-scratch-20perp-{temp}_metrics.parquet"
    metrics_1perp_parquet = f"_artifacts/mp-20-pxrd/temp/mp-20-scratch-1perp-{temp}_metrics.parquet"
    

    df_20perp = pd.read_parquet(metric_20perp_parquet)
    df_1perp = pd.read_parquet(metrics_1perp_parquet)

    results_20perp = get_metrics_xrd(df_20perp, n_test=1000, only_matched=False, verbose=False)
    results_1perp = get_metrics_xrd(df_1perp, n_test=1000, only_matched=False, verbose=False)
    results[temp] = (results_20perp, results_1perp)
    # print("\n")

table_20perp = pd.DataFrame.from_dict({temp: results[temp][0] for temp in results}, orient='index')
table_1perp = pd.DataFrame.from_dict({temp: results[temp][1] for temp in results}, orient='index')

# concatenate the two tables 20perp and 1perp one on top of the other
final_table = pd.concat([table_20perp, table_1perp], axis=0)
final_table.head(12)

#### 2. No XRD conditions, no scoring

> Note: the model without XRD conditioning is a plain GPT-2 trained without XRD inputs. It ignores the condition vectors filled with `-100` passed here.

In [ ]:
!python _utils/_generating/generate_CIFs.py --config '_config_files/generation/conditional/xrd_studies/ablation/mp-xrd-scratch-uncond-20shot_eval.jsonc'

In [ ]:
!python _utils/_generating/postprocess.py \
    --input_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-20shot_gen.parquet' \
    --output_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-20shot_post.parquet' \
    --num_workers 32 \
    --column_name 'Generated CIF'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-20shot_post.parquet' \
    --num_gens 20 \
    --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
    --output_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-20shot_metrics.parquet' \
    --num_workers 24 \
    --validity_check 'diffcsp'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-20shot_post.parquet' \
    --num_gens 1 \
    --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
    --output_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-1shot_metrics.parquet' \
    --num_workers 24 \
    --validity_check 'diffcsp'

#### 3. XRD conditions, no scoring

In [ ]:
!python _utils/_generating/generate_CIFs.py --config '_config_files/generation/conditional/xrd_studies/ablation/mp-xrd-scratch-20shot_eval.jsonc'

In [ ]:
!python _utils/_generating/postprocess.py \
    --input_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-20shot_gen.parquet' \
    --output_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-20shot_post.parquet' \
    --num_workers 32 \
    --column_name 'Generated CIF'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-20shot_post.parquet' \
    --num_gens 20 \
    --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
    --output_parquet '_artifacts/mp-20-pxrd/ablation/mp-20-scratch-20shot_metrics.parquet' \
    --num_workers 24 \
    --validity_check 'diffcsp'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/mp-20-pxrd/temp/mp-20-scratch-20perp-075_post.parquet' \
    --num_gens 1 \
    --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
    --output_parquet '_artifacts/mp-20-pxrd/temp/mp-20-scratch-1rand-075_metrics.parquet' \
    --num_workers 24 \
    --validity_check 'diffcsp' \
    --sort_gens 'random'

#### 4. Generation time
The ablation configurations are rerun sequentially on a single GPU to measure generation time. These runs supply only the timing column in the results table.

In [ ]:
import time
import pandas as pd

cfg_dir = "_config_files/generation/conditional/xrd_studies/ranking_time"
out = "_artifacts/mp-20-pxrd/ranking_time"
times = {}
for name in ["mp-cond-20cons", "mp-cond-20shot", "mp-cond-1shot", "mp-uncond-20shot", "mp-uncond-1shot"]:
    t0 = time.time()
    # one GPU only so the times are comparable
    !CUDA_VISIBLE_DEVICES=0 python _utils/_generating/generate_CIFs.py --config {cfg_dir}/{name}.jsonc
    times[name] = time.time() - t0

    !python _utils/_generating/postprocess.py \
    --input_parquet {out}/{name}_gen.parquet \
    --output_parquet {out}/{name}_post.parquet \
    --num_workers 16 \
    --column_name 'Generated CIF'

    for n in ([1] if "1shot" in name else [20, 1]):
        !python _utils/_metrics/XRD_metrics.py \
        --input_parquet {out}/{name}_post.parquet \
        --num_gens {n} \
        --ref_parquet '_artifacts/mp-20-pxrd/mp-test_ref.parquet' \
        --output_parquet {out}/{name}_n{n}_metrics.parquet \
        --num_workers 16 \
        --validity_check 'diffcsp'

pd.Series(times, name="seconds").rename_axis("run").to_csv(f"{out}/timings.csv")

### Results

In [ ]:
metrics_1perp_parquet = "_artifacts/mp-20-pxrd/temp/mp-20-scratch-1perp-075_metrics.parquet"
metrics_20perp_parquet = "_artifacts/mp-20-pxrd/temp/mp-20-scratch-20perp-075_metrics.parquet"
metrics_uncond_1shot_parquet = "_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-1shot_metrics.parquet"
metrics_uncond_20shot_parquet = "_artifacts/mp-20-pxrd/ablation/mp-20-scratch-uncond-20shot_metrics.parquet"
metrics_cond_1shot_parquet = "_artifacts/mp-20-pxrd/ablation/mp-20-scratch-1shot_metrics.parquet"
metrics_cond_20shot_parquet = "_artifacts/mp-20-pxrd/ablation/mp-20-scratch-20shot_metrics.parquet"

# make a table with all the results
import __init__
from _utils._notebook_utils.y_xrd_mp_20_ablations_utils import get_metrics_xrd
import pandas as pd
import numpy as np

paths = {
    'cond-20perp': metrics_20perp_parquet,
    'cond-1perp': metrics_1perp_parquet,
    'uncond-20shot': metrics_uncond_20shot_parquet,
    'uncond-1shot': metrics_uncond_1shot_parquet,
    'cond-20shot': metrics_cond_20shot_parquet,
    'cond-1shot': metrics_cond_1shot_parquet
}
results = {}

for names, path in paths.items():
    df = pd.read_parquet(path)
    metrics_result = get_metrics_xrd(df, n_test=1000, only_matched=False, verbose=False)
    results[names] = metrics_result
        
# Create final table with all results
final_table = pd.DataFrame.from_dict(results, orient='index')
final_table

In [ ]:
# generation time per target from the timed reruns (1-perplexity shares the 20-consistent run)
t = pd.read_csv("_artifacts/mp-20-pxrd/ranking_time/timings.csv", index_col="run")["seconds"] / 1000
final_table["Time (s/target)"] = pd.Series({
    "cond-20perp": t["mp-cond-20cons"],
    "cond-1perp": t["mp-cond-20cons"],
    "uncond-20shot": t["mp-uncond-20shot"],
    "uncond-1shot": t["mp-uncond-1shot"],
    "cond-20shot": t["mp-cond-20shot"],
    "cond-1shot": t["mp-cond-1shot"],
})
final_table